<a href="https://colab.research.google.com/github/abhisheknaik2k20/SMA_Project/blob/main/SMA_Experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import/Install Packages**

In [ ]:
!pip install supabase
from supabase import Client, create_client
from google.colab import userdata as env
from googleapiclient.discovery import build

# **POSTGRE DATABASE API CODE**

In [ ]:
class PostgreSQLDatabase:
  def __init__(self):
    try:
      self.supabase_client : Client = create_client(env.get('supabase_URL'), env.get('supabase_key'))
    except Exception as error:
      print(error)

  def get_genre_info(self):
    try:
      response=self.supabase_client.table('Genre').select('*', count ='exact').execute()
      return response.data, response.count
    except Exception as error:
      print(error)

  def populate_genre(self, data=[]):
    try:
      response=self.supabase_client.table('Genre').insert(data, count='exact').execute()
      return response.data, response.count
    except Exception as error:
      print(error)

  def populate_video(self, data=[]):
    try:
      response = self.supabase_client.table('Video').insert(data, count='exact').execute()
      return response.data, response.count
    except Exception as error:
      print(error)

  def get_video_info(self):
    try:
      response=self.supabase_client.table('Video').select('*', count ='exact').execute()
      return response.data, response.count
    except Exception as error:
      print(error)

  def return_video_ids(self):
    try:
      response= self.supabase_client.table('Video').select('video_id').execute()
      return [r['video_id'] for r in response.data]
    except Exception as error : print(error)

  def populate_comments(self, data=[]):
    try:
      response = self.supabase_client.table('Comments').insert(data, count='exact').execute()
      return response.data, response.count
    except Exception as error:
      print(error)

  def fetch_comments_data(self):
    try:
      response=self.supabase_client.table('Comments').select('*').execute()
      return response.data
    except Exception as error : print(error)

  def return_comment_ids(self):
    try:
      response= self.supabase_client.table('Comments').select('comment_id').execute()
      return [r['comment_id'] for r in response.data]
    except Exception as error : print(error)

  def return_video_ids_with_no_comment_data(self):
    try:
      video_ids = set(self.return_video_ids())
      commented_video_ids = set(r['video_id'] for r in self.supabase_client.rpc('get_distinct_comment_video_ids').execute().data)
      return list(video_ids.difference(commented_video_ids))
    except Exception as error : print(error)

# **YOUTUBE API CODE**

In [ ]:
class FetchYoutubeData:
  def __init__(self):
    try : self.youtube = build("youtube", "v3", developerKey=env.get('YT_v33') )
    except Exception as error :
      print(error)

  def fetch_video_ids(self, videoCategoryTag : int, order = 'viewCount'):
    try:
      search_response = self.youtube.search().list(part="id",type="video",videoCategoryId=videoCategoryTag, q=" ", order=order,relevanceLanguage="en",maxResults=20).execute()
      return [item["id"]["videoId"] for item in search_response["items"]]
    except Exception as error :
      print(error)

  def fetch_video_data(self,genre_id,video_ids):
    video_data=[]
    for video_id in video_ids:
      try:
        video_request = self.youtube.videos().list(part="snippet,contentDetails,statistics,status,topicDetails", id =  video_id).execute()
        item=video_request["items"][0]
        video_data.append({'video_id': item['id'],'title': item['snippet']['title'],'description': item['snippet']['description'],'published_at': item['snippet']['publishedAt'],'channel_id': item['snippet']['channelId'],'channel_title':item['snippet']['channelTitle'],'comment_count': int(item['statistics'].get('commentCount', 0)),'like_count': int(item['statistics'].get('likeCount', 0)),'view_count': int(item['statistics'].get('viewCount', 0)),'genre_id': genre_id})
      except Exception as error : continue
    return video_data

  def store_video_data(self):
    pgdb = PostgreSQLDatabase()
    responses, count = pgdb.get_genre_info()
    if count == -1 : return
    stored_video_data = set(pgdb.return_video_ids())
    for response in responses:
        video_ids = self.fetch_video_ids(response['video_tag'], order = 'relevance')
        if not video_ids : continue
        result = set(video_ids).difference(stored_video_data)
        if not result: continue
        stored_video_data.update(result)
        pgdb.populate_video(self.fetch_video_data(response['genre_id'], result))

  def fetch_comment_data(self, video_id):
    result=[]
    try:
      request = self.youtube.commentThreads().list(part="snippet",videoId=video_id,textFormat="plainText", order="relevance", maxResults=10).execute()
      for item in request.get("items", []):
        top_comment = item["snippet"]["topLevelComment"]["snippet"]
        result.append({"comment_id": item["id"],"video_id": video_id,"comment_text": top_comment["textDisplay"],"like_count": str(top_comment["likeCount"]),"comment_date": top_comment["publishedAt"][:10]})
      return result
    except Exception as error:
      return []

  def store_comment_data(self):
     pgdb = PostgreSQLDatabase()
     video_ids = pgdb.return_video_ids_with_no_comment_data()
     for video_id in video_ids:
      comments = self.fetch_comment_data(video_id)
      if not comments : continue
      pgdb.populate_comments(comments)

**Fetching and Storing Youtube Video Data**

In [ ]:
youtube = FetchYoutubeData()
youtube.store_video_data()

**Fetching and storing Youtube Video Comments Data**

In [ ]:
youtube = FetchYoutubeData()
youtube.store_comment_data()